<a href="https://colab.research.google.com/github/srikanthprabala/TorchCode/blob/master/templates/10_gqa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [9]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [10]:
import torch
import torch.nn as nn
import math

In [14]:
import torch
import torch.nn as nn
import math

class GroupQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        assert num_heads % num_kv_heads == 0, "num_heads must be divisible by num_kv_heads"

        self.num_heads = num_heads                # total Q heads
        self.num_kv_heads = num_kv_heads          # shared KV heads
        self.d_k = d_model // num_heads           # per-head dimension
        self.repeats = num_heads // num_kv_heads  # Q heads per KV head

        # Projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: [B, S, d_model]
        """

        B, S, _ = x.shape

        # -------------------------
        # 1. Project to Q, K, V
        # -------------------------
        # Q: [B, S, num_heads * d_k]
        # K/V: [B, S, num_kv_heads * d_k]
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        # -------------------------
        # 2. Reshape into heads
        # -------------------------
        # Q: [B, num_heads, S, d_k]
        q = q.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        # K, V: [B, num_kv_heads, S, d_k]
        k = k.view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)
        v = v.view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)

        # -------------------------
        # 3. Group Q heads
        # -------------------------
        # Instead of [B, num_heads, S, d_k]
        # we reshape into:
        # [B, num_kv_heads, repeats, S, d_k]
        q = q.view(B, self.num_kv_heads, self.repeats, S, self.d_k)

        # -------------------------
        # 4. Prepare K, V for broadcasting
        # -------------------------
        # Add a singleton dimension so they broadcast across repeats
        # K/V: [B, num_kv_heads, 1, S, d_k]
        k = k.unsqueeze(2)
        v = v.unsqueeze(2)

        # -------------------------
        # 5. Scaled dot-product attention
        # -------------------------
        # k^T: [B, num_kv_heads, 1, d_k, S]
        # scores: [B, num_kv_heads, repeats, S_q, S_k]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Softmax over key dimension
        weights = torch.softmax(scores, dim=-1)

        # -------------------------
        # 6. Apply attention to V
        # -------------------------
        # attn: [B, num_kv_heads, repeats, S, d_k]
        attn = torch.matmul(weights, v)

        # -------------------------
        # 7. Merge heads back
        # -------------------------
        # First reshape back to standard head layout:
        # [B, num_heads, S, d_k]
        attn = attn.view(B, self.num_heads, S, self.d_k)

        # Then:
        # [B, S, num_heads, d_k] → [B, S, d_model]
        out = attn.transpose(1, 2).contiguous().view(B, S, -1)

        # -------------------------
        # 8. Final output projection
        # -------------------------
        return self.W_o(out)

In [15]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [16]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.2ms)
  ✅ [2/5] nn.Linear with correct shapes (1.6ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (2.0ms)
  ✅ [4/5] KV heads are shared correctly (1.6ms)
  ✅ [5/5] Gradient flow (2.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (12.1ms total)
  Progress saved. Run status() to see your dashboard.

